In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
BRONZE_TABLE   = f"{CATALOG}.bronze.raw_nhl_game_summaries"
DIM_GAMES      = f"{CATALOG}.silver.dimGames"
DIM_ATHLETES   = f"{CATALOG}.silver.dimAthletes"
SILVER_TABLE   = f"{CATALOG}.silver.nhl_player_game_stats"
 
SPORT  = "hockey"
LEAGUE = "nhl"
 
# Fields already in nhl_skater_game_logs — skip these
SKIP_FIELDS = {
    "goals", "assists", "plusMinus", "shotsTotal", "penaltyMinutes",
    "timeOnIce", "ytdGoals", "shotsMissed", "shootoutGoals",
    "points", "shootingPct", "powerPlayGoals", "powerPlayAssists",
    "shortHandedGoals", "shortHandedAssists", "gameWinningGoals",
    "timeOnIcePerGame", "production",
}
 
print(f"Source    : {BRONZE_TABLE}")
print(f"Target    : {SILVER_TABLE}")

In [0]:
# ── READ SOURCES ──────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from datetime import datetime, timezone
import json
 
bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze rows : {bronze_df.count()}")
 
# ── dimGames reference ──
dim_games = (
    spark.table(DIM_GAMES)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_game_id"),
        F.col("source_event_id").alias("dim_event_id"),
    )
)
 
# ── dimAthletes reference — current active rows ──
dim_athletes = (
    spark.table(DIM_ATHLETES)
    .filter(
        (F.col("league") == LEAGUE) &
        (F.col("is_current") == True)
    )
    .select(
        F.col("athlete_id").cast("string").alias("dim_athlete_id"),
        F.col("clutch_athlete_id"),
        F.col("clutch_team_id"),
    )
)
 
print(f"dimGames ({LEAGUE})   : {dim_games.count()}")
print(f"dimAthletes ({LEAGUE}): {dim_athletes.count()}")

In [0]:
# ── PARSE PLAYER STATS FROM BOXSCORE ─────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
skipped     = []
 
def parse_toi(toi_str):
    """Convert MM:SS string to integer seconds. Returns 0 if unparseable."""
    try:
        if not toi_str or ":" not in str(toi_str):
            return 0
        parts = str(toi_str).split(":")
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return 0
 
def to_int(val):
    try:
        return int(float(val)) if val not in (None, "", "null") else None
    except:
        return None
 
def to_float(val):
    try:
        return float(val) if val not in (None, "", "null") else None
    except:
        return None
 
bronze_rows = bronze_df.collect()
 
for br in bronze_rows:
    event_id    = br["event_id"]
    season      = br["season"]
    season_type = br["season_type"]
    round_num   = br["round"]
 
    try:
        boxscore       = json.loads(br["boxscore_json"])
        players_blocks = boxscore.get("players", [])
    except Exception as e:
        skipped.append((event_id, str(e)))
        continue
 
    for team_players in players_blocks:
        team_info = team_players.get("team", {})
        team_id   = team_info.get("id")
        team_abbr = team_info.get("abbreviation")
 
        for stat_group in team_players.get("statistics", []):
            stat_group_name = stat_group.get("name", "unknown")
            keys            = stat_group.get("keys", [])
            athletes        = stat_group.get("athletes", [])
 
            for athlete_entry in athletes:
                athlete    = athlete_entry.get("athlete", {})
                athlete_id = athlete.get("id")
                stats_arr  = athlete_entry.get("stats", [])
                scratched  = athlete.get("scratched", False)
 
                # ── Zip keys with stats ──
                stat_dict = dict(zip(keys, stats_arr))
 
                # ── Extract only net-new fields ──
                rows.append({
                    # ── Natural keys ──
                    "athlete_id":       str(athlete_id) if athlete_id else None,
                    "event_id":         event_id,
                    "team_id":          team_id,
 
                    # ── Context ──
                    "team_abbreviation": team_abbr,
                    "stat_group":       stat_group_name,
                    "sport":            SPORT,
                    "league":           LEAGUE,
                    "season":           season,
                    "season_type":      season_type,
                    "round":            round_num,
 
                    # ── Participation ──
                    "scratched":        scratched,
 
                    # ── Additive stats (net new vs nhl_skater_game_logs) ──
                    "blocked_shots":    to_int(stat_dict.get("blockedShots")),
                    "hits":             to_int(stat_dict.get("hits")),
                    "takeaways":        to_int(stat_dict.get("takeaways")),
                    "giveaways":        to_int(stat_dict.get("giveaways")),
                    "shifts":           to_int(stat_dict.get("shifts")),
 
                    # ── TOI splits (MM:SS → seconds) ──
                    "pp_toi_seconds":   parse_toi(stat_dict.get("powerPlayTimeOnIce")),
                    "sh_toi_seconds":   parse_toi(stat_dict.get("shortHandedTimeOnIce")),
                    "es_toi_seconds":   parse_toi(stat_dict.get("evenStrengthTimeOnIce")),
 
                    # ── Faceoffs (centers primarily) ──
                    "faceoffs_won":     to_int(stat_dict.get("faceoffsWon")),
                    "faceoffs_lost":    to_int(stat_dict.get("faceoffsLost")),
                    "faceoff_pct":      to_float(stat_dict.get("faceoffPercent")),
 
                    # ── Metadata ──
                    "ingested_at":      ingested_at,
                    "source_table":     "bronze.raw_nhl_game_summaries",
                })
 
print(f"Rows built : {len(rows)}")
print(f"Skipped    : {len(skipped)}")
if skipped:
    for event_id, reason in skipped:
        print(f"  event {event_id}: {reason}")

In [0]:
# ── SPOT CHECK ────────────────────────────────────────────────────────────────
 
if rows:
    sample = rows[0]
    print(f"Sample row — {sample['team_abbreviation']} | {sample['stat_group']}")
    print(f"  athlete_id    : {sample['athlete_id']}")
    print(f"  blocked_shots : {sample['blocked_shots']}")
    print(f"  hits          : {sample['hits']}")
    print(f"  takeaways     : {sample['takeaways']}")
    print(f"  giveaways     : {sample['giveaways']}")
    print(f"  shifts        : {sample['shifts']}")
    print(f"  pp_toi_seconds: {sample['pp_toi_seconds']}")
    print(f"  sh_toi_seconds: {sample['sh_toi_seconds']}")
    print(f"  es_toi_seconds: {sample['es_toi_seconds']}")
    print(f"  faceoffs_won  : {sample['faceoffs_won']}")
    print(f"  faceoff_pct   : {sample['faceoff_pct']}")
    print(f"  scratched     : {sample['scratched']}")
 
    # Stat group breakdown
    from collections import Counter
    group_counts = Counter(r["stat_group"] for r in rows)
    print(f"\nStat groups:")
    for group, count in sorted(group_counts.items()):
        print(f"  {group:<20} : {count} rows")

In [0]:
# ── BUILD DATAFRAME + JOIN DIM REFERENCES ────────────────────────────────────
 
player_stats_df = spark.createDataFrame(rows)
 
# ── Join dimGames → clutch_game_id ──
player_stats_df = (
    player_stats_df
    .join(
        dim_games,
        player_stats_df.event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Join dimAthletes → clutch_athlete_id ──
player_stats_df = (
    player_stats_df
    .join(
        dim_athletes,
        player_stats_df.athlete_id == dim_athletes.dim_athlete_id,
        how="left"
    )
    .drop("dim_athlete_id")
)
 
# ── Warn on unmatched joins ──
unmatched_games    = player_stats_df.filter(F.col("clutch_game_id").isNull()).count()
unmatched_athletes = player_stats_df.filter(F.col("clutch_athlete_id").isNull()).count()
print(f"Unmatched dimGames    : {unmatched_games}    {'✓' if unmatched_games == 0 else '<-- investigate'}")
print(f"Unmatched dimAthletes : {unmatched_athletes} {'✓' if unmatched_athletes == 0 else '<-- note: some players may not be on playoff rosters'}")
 
# ── Final column order ──
player_stats_df = player_stats_df.select(
    # ── Surrogate FKs ──
    "clutch_game_id",
    "clutch_athlete_id",
    "clutch_team_id",
 
    # ── Natural keys ──
    "athlete_id",
    "event_id",
    "team_id",
 
    # ── Context ──
    "team_abbreviation",
    "stat_group",
    "sport",
    "league",
    "season",
    "season_type",
    "round",
 
    # ── Participation ──
    "scratched",
 
    # ── Additive stats ──
    "blocked_shots",
    "hits",
    "takeaways",
    "giveaways",
    "shifts",
 
    # ── TOI splits ──
    "pp_toi_seconds",
    "sh_toi_seconds",
    "es_toi_seconds",
 
    # ── Faceoffs ──
    "faceoffs_won",
    "faceoffs_lost",
    "faceoff_pct",
 
    # ── Metadata ──
    "ingested_at",
    "source_table",
)
 
total_rows = player_stats_df.count()
print(f"\nTotal rows to write: {total_rows}")

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# MERGE on athlete_id + event_id — safe for re-runs and new round uploads.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        player_stats_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    player_stats_df.createOrReplaceTempView("new_player_stats")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_player_stats AS source
        ON  target.athlete_id = source.athlete_id
        AND target.event_id   = source.event_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Sample — first 10 rows ──")
spark.sql(f"""
    SELECT
        clutch_game_id,
        clutch_athlete_id,
        athlete_id,
        event_id,
        team_abbreviation,
        stat_group,
        scratched,
        blocked_shots,
        hits,
        takeaways,
        giveaways,
        shifts,
        pp_toi_seconds,
        es_toi_seconds,
        faceoffs_won,
        faceoff_pct
    FROM {SILVER_TABLE}
    ORDER BY event_id, stat_group, athlete_id
    LIMIT 10
""").show(10, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(DISTINCT event_id)                                        AS unique_games,
        COUNT(DISTINCT athlete_id)                                      AS unique_athletes,
        COUNT(CASE WHEN clutch_game_id IS NULL    THEN 1 END)          AS null_clutch_game_ids,
        COUNT(CASE WHEN clutch_athlete_id IS NULL THEN 1 END)          AS null_clutch_athlete_ids,
        COUNT(CASE WHEN scratched = true          THEN 1 END)          AS scratched_players,
        COUNT(CASE WHEN stat_group = 'forwards'   THEN 1 END)          AS forward_rows,
        COUNT(CASE WHEN stat_group = 'defense'    THEN 1 END)          AS defense_rows,
        COUNT(CASE WHEN stat_group = 'goalies'    THEN 1 END)          AS goalie_rows,
        COUNT(CASE WHEN blocked_shots IS NULL     THEN 1 END)          AS null_blocked_shots,
        COUNT(CASE WHEN faceoffs_won > 0          THEN 1 END)          AS players_with_faceoffs,
        ROUND(AVG(CASE WHEN stat_group != 'goalies' AND scratched = false
                       THEN shifts END), 1)                             AS avg_shifts_skaters,
        ROUND(AVG(CASE WHEN stat_group != 'goalies' AND scratched = false
                       THEN es_toi_seconds END), 0)                     AS avg_es_toi_seconds
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)